In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# 1. Load and Split Data
df = sns.load_dataset('tips')
X = df.drop('total_bill', axis=1)
y = df['total_bill']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

# 2. Preprocessing Pipelines
cat_cols = ["sex", "smoker", "day", "time"]
num_cols = ["tip", "size"]

num_pipeline = Pipeline(steps=[
    ('imputation', SimpleImputer(strategy='median')),
    ('scaling', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputation', SimpleImputer(strategy='most_frequent')),
    ('encoding', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ("num_pipeline", num_pipeline, num_cols),
    ("cat_pipeline", cat_pipeline, cat_cols)
])

# 3. Transform Data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# 4. Hyperparameter Tuning for Random Forest
# Note: Using regression-specific criteria and scoring
params = {
    'max_depth': [3, 5, 10, None],
    'n_estimators': [50, 100, 200, 300],
    'criterion': ['squared_error', 'absolute_error'] 
}

rfr_cv = RandomizedSearchCV(
    RandomForestRegressor(), 
    param_distributions=params, 
    n_iter=10, 
    cv=5, 
    scoring='r2', 
    random_state=1
)

# 5. Define Models Dictionary
models = {
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(),
    "Linear Regression": LinearRegression(),
    "Random Forest Default": RandomForestRegressor(),
    "Random Forest Tuned": rfr_cv
}

# 6. Training and Evaluation Function
def train_pred_eval(X_train, X_test, y_train, y_test, models_dict):
    results = {}
    for name, model in models_dict.items():
        # Train
        model.fit(X_train, y_train)
        # Predict
        y_pred = model.predict(X_test)
        
        # Calculate Metrics
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Store results in a readable format
        results[name] = {
            "MSE": round(mse, 4),
            "R2_Score": round(r2, 4)
        }
    return results

# 7. Execute and View Results
evaluation_results = train_pred_eval(X_train, X_test, y_train, y_test, models)

# Convert to DataFrame for better visualization
results_df = pd.DataFrame(evaluation_results).T
print(results_df)

                           MSE  R2_Score
SVR                    54.4172    0.3429
Decision Tree          76.0674    0.0815
Linear Regression      41.1314    0.5033
Random Forest Default  47.9676    0.4208
Random Forest Tuned    46.4872    0.4387
